# 02b — Algorithm Shootout: Thematic Similarity Benchmarks

**Builds on:** `02_dynamic_recommendation_layer.ipynb` (Phase 2)

---

### Purpose

Phase 2 used Jaccard similarity on small tag sets to gate thematic relevance.
This notebook benchmarks **three distinct algorithms** on an expanded tag taxonomy
(10–15 tags per location) to determine which produces the most realistic
thematic-similarity signal for the Decision Support System.

| Algorithm | Type | Mechanism |
|---|---|---|
| **TF-IDF + Cosine** | NLP-inspired | Weights rare tags higher via inverse document frequency |
| **Unsupervised KNN** | Metric-learning | Combines binary tag features with normalised numerical scores |
| **Dice Coefficient** | Set-theoretic baseline | Pure set overlap, penalises size asymmetry less than Jaccard |

---

### Pipeline

```
30-Location Dataset (10–15 tags each)
        │
        ├─▶ [TF-IDF + Cosine]   → tfidf_cosine_score (0–100)
        ├─▶ [KNN Similarity]    → knn_similarity_score (0–100)
        └─▶ [Dice Coefficient]  → dice_score (0–100)
        │
        ▼
   compare_algorithms(anchor) → Styled DataFrame
```

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

print("All imports loaded successfully.")

All imports loaded successfully.


---
## Step 1 — Expanded 30-Location Dataset

Each location now carries **10–15 highly detailed tags** organised into three conceptual tiers:

| Tier | Purpose | Examples |
|---|---|---|
| **Core Motivators** | Primary draw that drives visitation | `temple`, `hot_spring`, `urban` |
| **Vibe** | Atmospheric / experiential character | `serene`, `lively`, `rustic` |
| **Micro-features** | Specific activities or assets | `matcha`, `sake_brewery`, `cable_car` |

Tags are stored as **lists** (not sets) for compatibility with `MultiLabelBinarizer` and `TfidfVectorizer`.

In [2]:
# ── 30-Location Extended Japan Tourism Dataset (Expanded Tags) ──────────────
data = [
    # ═══════════════════════  MAJOR ANCHORS  ═══════════════════════
    {"region": "Tokyo", "is_anchor": True, "tourist_count": 200000, "capacity": 150000,
     "avg_delay_minutes": 25, "cultural_score": 85, "experiential_score": 95,
     "infrastructure_score": 95, "promotion_score": 100, "cleanliness_score": 80,
     "tags": ["urban", "modern", "shopping", "food", "nightlife", "anime",
             "tech", "skyscraper", "street_food", "museum", "lively",
             "fashion", "train_hub", "neon", "pop_culture"]},

    {"region": "Kyoto", "is_anchor": True, "tourist_count": 120000, "capacity": 80000,
     "avg_delay_minutes": 30, "cultural_score": 95, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 95, "cleanliness_score": 72,
     "tags": ["historic", "temple", "shrine", "cherry_blossom", "traditional",
             "unesco", "culture", "geisha", "gardens", "bamboo",
             "matcha", "photo_spot", "architecture", "autumn_leaves", "crafts"]},

    {"region": "Osaka", "is_anchor": True, "tourist_count": 150000, "capacity": 120000,
     "avg_delay_minutes": 20, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 90, "promotion_score": 90, "cleanliness_score": 75,
     "tags": ["urban", "food", "nightlife", "modern", "street_food",
             "castle", "lively", "comedy", "shopping", "takoyaki",
             "neon", "canal", "entertainment"]},

    {"region": "Hakone", "is_anchor": True, "tourist_count": 60000, "capacity": 40000,
     "avg_delay_minutes": 35, "cultural_score": 85, "experiential_score": 90,
     "infrastructure_score": 80, "promotion_score": 85, "cleanliness_score": 78,
     "tags": ["hot_spring", "nature", "mountain", "view", "lake",
             "ryokan", "cable_car", "volcanic", "serene", "art_museum",
             "scenic_train", "relaxation", "fuji_view"]},

    {"region": "Nara", "is_anchor": True, "tourist_count": 70000, "capacity": 50000,
     "avg_delay_minutes": 25, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 80, "promotion_score": 80, "cleanliness_score": 74,
     "tags": ["historic", "temple", "nature", "deer", "unesco",
             "traditional", "park", "buddha", "ancient", "serene",
             "photo_spot", "culture", "architecture"]},

    {"region": "Hiroshima", "is_anchor": True, "tourist_count": 90000, "capacity": 70000,
     "avg_delay_minutes": 20, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 75, "promotion_score": 85, "cleanliness_score": 78,
     "tags": ["historic", "memorial", "peace", "temple", "island",
             "unesco", "culture", "coastal", "tram", "okonomiyaki",
             "architecture", "museum", "solemn"]},

    {"region": "Kamakura", "is_anchor": True, "tourist_count": 55000, "capacity": 40000,
     "avg_delay_minutes": 28, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 80, "cleanliness_score": 75,
     "tags": ["historic", "temple", "coastal", "buddha", "shrine",
             "hiking", "beach", "traditional", "serene", "photo_spot",
             "architecture", "crafts", "day_trip"]},

    {"region": "Nikko", "is_anchor": True, "tourist_count": 45000, "capacity": 35000,
     "avg_delay_minutes": 30, "cultural_score": 92, "experiential_score": 88,
     "infrastructure_score": 70, "promotion_score": 75, "cleanliness_score": 80,
     "tags": ["historic", "shrine", "mountain", "nature", "unesco",
             "waterfall", "autumn_leaves", "ornate", "cedar", "lake",
             "hiking", "traditional", "architecture"]},

    {"region": "Sapporo", "is_anchor": True, "tourist_count": 80000, "capacity": 70000,
     "avg_delay_minutes": 15, "cultural_score": 75, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 85, "cleanliness_score": 85,
     "tags": ["urban", "snow", "food", "festival", "beer",
             "ski", "ramen", "park", "modern", "winter_sport",
             "lively", "night_market", "scenic"]},

    {"region": "Fukuoka", "is_anchor": True, "tourist_count": 65000, "capacity": 60000,
     "avg_delay_minutes": 10, "cultural_score": 80, "experiential_score": 90,
     "infrastructure_score": 88, "promotion_score": 80, "cleanliness_score": 82,
     "tags": ["urban", "food", "historic", "shopping", "ramen",
             "shrine", "night_market", "canal", "modern", "lively",
             "street_food", "port"]},

    # ═══════════════════════  ORBIT DESTINATIONS  ═══════════════════════
    {"region": "Kanazawa", "is_anchor": False, "tourist_count": 25000, "capacity": 50000,
     "avg_delay_minutes": 8, "cultural_score": 85, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 60, "cleanliness_score": 88,
     "tags": ["historic", "cherry_blossom", "garden", "traditional", "crafts",
             "geisha", "samurai", "museum", "culture", "serene",
             "architecture", "matcha", "autumn_leaves"]},

    {"region": "Takayama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 80, "experiential_score": 75,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 90,
     "tags": ["alpine", "mountain", "rural", "historic", "traditional",
             "festival", "sake_brewery", "edo_town", "crafts", "serene",
             "hiking", "wood_carving", "morning_market"]},

    {"region": "Shirakawa-go", "is_anchor": False, "tourist_count": 22000, "capacity": 20000,
     "avg_delay_minutes": 15, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 50, "promotion_score": 70, "cleanliness_score": 80,
     "tags": ["rural", "historic", "snow", "mountain", "unesco",
             "traditional", "thatched_roof", "village", "photo_spot", "serene",
             "farming", "culture", "winter"]},

    {"region": "Koyasan", "is_anchor": False, "tourist_count": 12000, "capacity": 25000,
     "avg_delay_minutes": 5, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 60, "promotion_score": 50, "cleanliness_score": 85,
     "tags": ["temple", "historic", "mountain", "rural", "unesco",
             "buddhist", "cemetery", "serene", "meditation", "cable_car",
             "traditional", "spiritual", "pilgrim", "shojin_ryori"]},

    {"region": "Nagasaki", "is_anchor": False, "tourist_count": 20000, "capacity": 45000,
     "avg_delay_minutes": 8, "cultural_score": 88, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 82,
     "tags": ["historic", "memorial", "coastal", "food", "tram",
             "culture", "church", "port", "island", "museum",
             "solemn", "international", "night_view"]},

    {"region": "Tottori", "is_anchor": False, "tourist_count": 12000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 70, "experiential_score": 78,
     "infrastructure_score": 60, "promotion_score": 40, "cleanliness_score": 85,
     "tags": ["rural", "nature", "sand_dunes", "coastal", "adventure",
             "camel_ride", "paragliding", "museum", "quiet", "scenic",
             "seafood", "off_beaten_path"]},

    {"region": "Akita", "is_anchor": False, "tourist_count": 9000, "capacity": 30000,
     "avg_delay_minutes": 3, "cultural_score": 65, "experiential_score": 72,
     "infrastructure_score": 55, "promotion_score": 35, "cleanliness_score": 82,
     "tags": ["rural", "nature", "festival", "mountain", "snow",
             "onsen", "lake", "samurai", "rice_paddy", "quiet",
             "traditional", "rustic"]},

    {"region": "Shimane", "is_anchor": False, "tourist_count": 7000, "capacity": 25000,
     "avg_delay_minutes": 2, "cultural_score": 75, "experiential_score": 70,
     "infrastructure_score": 50, "promotion_score": 30, "cleanliness_score": 87,
     "tags": ["historic", "shrine", "garden", "rural", "mythology",
             "traditional", "serene", "castle", "quiet", "culture",
             "off_beaten_path", "sunset"]},

    {"region": "Okayama", "is_anchor": False, "tourist_count": 15000, "capacity": 40000,
     "avg_delay_minutes": 5, "cultural_score": 82, "experiential_score": 75,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 86,
     "tags": ["historic", "garden", "castle", "culture", "traditional",
             "crafts", "denim", "fruit", "cycling", "serene",
             "architecture", "museum"]},

    {"region": "Kagawa", "is_anchor": False, "tourist_count": 14000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 82,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 84,
     "tags": ["food", "coastal", "shrine", "udon", "island",
             "art", "garden", "pilgrimage", "scenic", "cycling",
             "bridge", "traditional"]},

    {"region": "Naoshima", "is_anchor": False, "tourist_count": 11000, "capacity": 15000,
     "avg_delay_minutes": 12, "cultural_score": 88, "experiential_score": 90,
     "infrastructure_score": 55, "promotion_score": 60, "cleanliness_score": 88,
     "tags": ["art", "coastal", "modern", "rural", "museum",
             "architecture", "island", "sculpture", "gallery", "photo_spot",
             "minimalist", "design", "cycling"]},

    {"region": "Matsuyama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 85, "experiential_score": 82,
     "infrastructure_score": 72, "promotion_score": 50, "cleanliness_score": 85,
     "tags": ["hot_spring", "historic", "castle", "tram", "traditional",
             "literature", "garden", "relaxation", "food", "serene",
             "architecture", "culture"]},

    {"region": "Beppu", "is_anchor": False, "tourist_count": 25000, "capacity": 45000,
     "avg_delay_minutes": 10, "cultural_score": 75, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 80,
     "tags": ["hot_spring", "nature", "coastal", "volcanic", "steam",
             "relaxation", "ryokan", "sand_bath", "scenic", "food",
             "hell_tour", "unique"]},

    {"region": "Kurokawa Onsen", "is_anchor": False, "tourist_count": 8000, "capacity": 12000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 45, "promotion_score": 40, "cleanliness_score": 92,
     "tags": ["hot_spring", "rural", "mountain", "ryokan", "serene",
             "traditional", "nature", "relaxation", "lantern", "rustic",
             "intimate", "off_beaten_path"]},

    {"region": "Kagoshima", "is_anchor": False, "tourist_count": 16000, "capacity": 38000,
     "avg_delay_minutes": 5, "cultural_score": 78, "experiential_score": 85,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 83,
     "tags": ["nature", "coastal", "mountain", "hot_spring", "volcanic",
             "scenic", "food", "garden", "historic", "tram",
             "island", "adventure", "subtropical"]},

    {"region": "Sendai", "is_anchor": False, "tourist_count": 30000, "capacity": 65000,
     "avg_delay_minutes": 8, "cultural_score": 82, "experiential_score": 80,
     "infrastructure_score": 85, "promotion_score": 60, "cleanliness_score": 84,
     "tags": ["urban", "historic", "food", "festival", "castle",
             "shopping", "beef_tongue", "tanabata", "park", "modern",
             "day_trip", "lively"]},

    {"region": "Aomori", "is_anchor": False, "tourist_count": 10000, "capacity": 35000,
     "avg_delay_minutes": 3, "cultural_score": 75, "experiential_score": 80,
     "infrastructure_score": 60, "promotion_score": 45, "cleanliness_score": 86,
     "tags": ["nature", "snow", "rural", "festival", "nebuta",
             "apple", "hiking", "mountain", "lake", "scenic",
             "autumn_leaves", "quiet", "rustic"]},

    {"region": "Nagano", "is_anchor": False, "tourist_count": 22000, "capacity": 50000,
     "avg_delay_minutes": 7, "cultural_score": 85, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 60, "cleanliness_score": 87,
     "tags": ["mountain", "snow", "nature", "temple", "ski",
             "monkey_park", "hiking", "alpine", "traditional", "soba",
             "scenic", "onsen", "winter_sport"]},

    {"region": "Matsumoto", "is_anchor": False, "tourist_count": 15000, "capacity": 35000,
     "avg_delay_minutes": 5, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 55, "cleanliness_score": 88,
     "tags": ["castle", "historic", "alpine", "traditional", "culture",
             "crafts", "soba", "museum", "architecture", "serene",
             "mountain", "wasabi", "art"]},

    {"region": "Toyama", "is_anchor": False, "tourist_count": 13000, "capacity": 40000,
     "avg_delay_minutes": 4, "cultural_score": 75, "experiential_score": 85,
     "infrastructure_score": 68, "promotion_score": 50, "cleanliness_score": 85,
     "tags": ["alpine", "nature", "coastal", "snow", "scenic",
             "glass_art", "sushi", "firefly_squid", "mountain", "dam",
             "hiking", "modern", "quiet"]},
]

df = pd.DataFrame(data)

# Quick validation
tag_counts = df["tags"].apply(len)
print(f"Locations: {len(df)}  |  Anchors: {df['is_anchor'].sum()}  |  Orbits: {(~df['is_anchor']).sum()}")
print(f"Tags per location — min: {tag_counts.min()}, max: {tag_counts.max()}, mean: {tag_counts.mean():.1f}")
df[["region", "is_anchor", "tags"]].head(10)

Locations: 30  |  Anchors: 10  |  Orbits: 20
Tags per location — min: 12, max: 15, mean: 12.8


,region,is_anchor,tags
0,Tokyo,True,"[urban, modern, shopping, food, nightlife, ani..."
1,Kyoto,True,"[historic, temple, shrine, cherry_blossom, tra..."
2,Osaka,True,"[urban, food, nightlife, modern, street_food, ..."
3,Hakone,True,"[hot_spring, nature, mountain, view, lake, ryo..."
4,Nara,True,"[historic, temple, nature, deer, unesco, tradi..."
5,Hiroshima,True,"[historic, memorial, peace, temple, island, un..."
6,Kamakura,True,"[historic, temple, coastal, buddha, shrine, hi..."
7,Nikko,True,"[historic, shrine, mountain, nature, unesco, w..."
8,Sapporo,True,"[urban, snow, food, festival, beer, ski, ramen..."
9,Fukuoka,True,"[urban, food, historic, shopping, ramen, shrin..."


---
## Algorithm 1 — TF-IDF + Cosine Similarity

**Concept:** Treat each location's tag list as a "document" of space-separated terms.
`TfidfVectorizer` produces a term-frequency × inverse-document-frequency matrix
that **up-weights rare, distinctive tags** (e.g. `geisha`, `firefly_squid`) and
**down-weights common tags** shared by many locations (e.g. `historic`, `nature`).

Cosine similarity then measures the angle between the TF-IDF vectors of two locations — a value of 1.0 means identical tag profiles; 0.0 means no overlap after IDF weighting.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

The result is scaled to 0–100 for comparability.

In [3]:
# ── Algorithm 1: TF-IDF + Cosine Similarity ─────────────────────────────────
def compute_tfidf_cosine_scores(df: pd.DataFrame, anchor_name: str) -> pd.DataFrame:
    """
    Compute TF-IDF cosine similarity between `anchor_name` and every other location.
    Returns a DataFrame with columns: region, tfidf_cosine_score (0-100).
    """
    # Join tags into single strings (TfidfVectorizer expects text)
    tag_strings = df["tags"].apply(lambda t: " ".join(t))

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(tag_strings)

    # Locate the anchor row index
    anchor_idx = df.index[df["region"] == anchor_name][0]
    anchor_vec = tfidf_matrix[anchor_idx]

    # Cosine similarity: anchor vs all
    cos_sim = cosine_similarity(anchor_vec, tfidf_matrix).flatten()

    result = pd.DataFrame({
        "region": df["region"],
        "tfidf_cosine_score": np.round(cos_sim * 100, 2),
    })

    # Exclude the anchor itself
    result = result[result["region"] != anchor_name].reset_index(drop=True)
    return result


# Quick test
tfidf_result = compute_tfidf_cosine_scores(df, "Kyoto")
tfidf_result.sort_values("tfidf_cosine_score", ascending=False).head(5)

,region,tfidf_cosine_score
9,Kanazawa,61.09
5,Kamakura,35.12
3,Nara,32.45
6,Nikko,28.58
4,Hiroshima,21.80


---
## Algorithm 2 — Unsupervised KNN (K-Nearest Neighbors)

**Concept:** Rather than treating tags as text, we create a **binary feature matrix**
via `MultiLabelBinarizer` (one column per unique tag, 0/1 per location). We then
concatenate normalised numerical scores (cultural, experiential, cleanliness, infrastructure)
to capture richer destination profiles.

`NearestNeighbors` with `metric='cosine'` finds the nearest destinations in this
combined feature space. Cosine distance is converted to a 0–100 similarity:

$$\text{knn\_similarity} = (1 - \text{cosine\_distance}) \times 100$$

**Key difference from TF-IDF:** KNN uses raw binary tag presence (no IDF weighting)
but compensates by incorporating numerical quality signals that TF-IDF ignores entirely.

In [4]:
# ── Algorithm 2: Unsupervised KNN ────────────────────────────────────────────
def compute_knn_similarity_scores(df: pd.DataFrame, anchor_name: str) -> pd.DataFrame:
    """
    Compute KNN cosine-distance-based similarity between `anchor_name`
    and every other location, using binary tag features + normalised numerics.
    Returns a DataFrame with columns: region, knn_similarity_score (0-100).
    """
    # ── Binary tag matrix ──────────────────────────────────────────────────
    mlb = MultiLabelBinarizer()
    tag_matrix = mlb.fit_transform(df["tags"])

    # ── Normalised numerical features ─────────────────────────────────────
    num_cols = ["cultural_score", "experiential_score",
               "cleanliness_score", "infrastructure_score"]
    scaler = MinMaxScaler()
    num_matrix = scaler.fit_transform(df[num_cols].values)

    # ── Combined feature matrix ───────────────────────────────────────────
    combined = np.hstack([tag_matrix, num_matrix])

    # ── Fit KNN ───────────────────────────────────────────────────────────
    knn = NearestNeighbors(n_neighbors=len(df), metric="cosine")
    knn.fit(combined)

    anchor_idx = df.index[df["region"] == anchor_name][0]
    distances, indices = knn.kneighbors(combined[anchor_idx].reshape(1, -1))

    # Convert cosine distance → similarity (0-100)
    similarities = (1 - distances.flatten()) * 100
    neighbor_regions = df.iloc[indices.flatten()]["region"].values

    result = pd.DataFrame({
        "region": neighbor_regions,
        "knn_similarity_score": np.round(similarities, 2),
    })

    # Exclude the anchor itself
    result = result[result["region"] != anchor_name].reset_index(drop=True)
    return result


# Quick test
knn_result = compute_knn_similarity_scores(df, "Kyoto")
knn_result.sort_values("knn_similarity_score", ascending=False).head(5)

,region,knn_similarity_score
0,Kanazawa,65.63
1,Nara,56.41
2,Kamakura,55.67
3,Nikko,49.35
4,Hiroshima,41.92


---
## Algorithm 3 — Dice Coefficient (Baseline)

**Concept:** The Dice coefficient is a set-similarity metric closely related to Jaccard,
but it gives **double weight to intersections** and is generally ≥ Jaccard for the same pair:

$$\text{Dice}(A, B) = \frac{2 \times |A \cap B|}{|A| + |B|}$$

Unlike TF-IDF, Dice treats all tags equally — `historic` and `geisha` contribute identically.
This makes it a useful **baseline**: any algorithm that fails to outperform Dice on
ranking quality is adding complexity without value.

In [5]:
# ── Algorithm 3: Dice Coefficient ────────────────────────────────────────────
def dice_coefficient(set_a: set, set_b: set) -> float:
    """
    Dice coefficient: (2 * |A ∩ B|) / (|A| + |B|).
    Returns 0.0 if both sets are empty.
    """
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a & set_b)
    return (2 * intersection) / (len(set_a) + len(set_b))


def compute_dice_scores(df: pd.DataFrame, anchor_name: str) -> pd.DataFrame:
    """
    Compute Dice coefficient between `anchor_name` and every other location.
    Returns a DataFrame with columns: region, shared_tags_count, dice_score (0-100).
    """
    anchor_tags = set(df.loc[df["region"] == anchor_name, "tags"].values[0])

    rows = []
    for _, row in df.iterrows():
        if row["region"] == anchor_name:
            continue
        orbit_tags = set(row["tags"])
        shared = anchor_tags & orbit_tags
        dice = dice_coefficient(anchor_tags, orbit_tags)
        rows.append({
            "region": row["region"],
            "shared_tags_count": len(shared),
            "dice_score": round(dice * 100, 2),
        })

    return pd.DataFrame(rows)


# Quick test
dice_result = compute_dice_scores(df, "Kyoto")
dice_result.sort_values("dice_score", ascending=False).head(5)

,region,shared_tags_count,dice_score
9,Kanazawa,9,64.29
5,Kamakura,7,50.00
3,Nara,7,50.00
6,Nikko,6,42.86
17,Okayama,5,37.04


---
## The Shootout — `compare_algorithms(anchor_name)`

This function runs all three algorithms against the same anchor and merges results
into a single styled DataFrame.

**No congestion or feasibility penalties are applied** — this is a pure thematic-similarity benchmark.

| Column | Source |
|---|---|
| `shared_tags_count` | Raw tag-set intersection count |
| `dice_score` | Algorithm 3 — Dice Coefficient (0–100) |
| `tfidf_cosine_score` | Algorithm 1 — TF-IDF + Cosine (0–100) |
| `knn_similarity_score` | Algorithm 2 — KNN Cosine distance (0–100) |

In [6]:
# ── The Shootout Function ─────────────────────────────────────────────────────
def compare_algorithms(anchor_name: str) -> pd.DataFrame:
    """
    Run all three similarity algorithms for a given anchor and return
    a merged, styled DataFrame sorted by tfidf_cosine_score DESC.
    """
    # ── Run each algorithm ──────────────────────────────────────────────
    dice_df  = compute_dice_scores(df, anchor_name)
    tfidf_df = compute_tfidf_cosine_scores(df, anchor_name)
    knn_df   = compute_knn_similarity_scores(df, anchor_name)

    # ── Merge on region ─────────────────────────────────────────────────
    merged = dice_df.merge(tfidf_df, on="region").merge(knn_df, on="region")

    # ── Sort by TF-IDF cosine score descending ─────────────────────────
    merged = (
        merged.sort_values("tfidf_cosine_score", ascending=False)
        .reset_index(drop=True)
    )

    # ── Column order ───────────────────────────────────────────────────
    col_order = ["region", "shared_tags_count", "dice_score",
                 "tfidf_cosine_score", "knn_similarity_score"]
    merged = merged[col_order]

    return merged


print("compare_algorithms() ready.")

compare_algorithms() ready.


In [7]:
# ── Run the Shootout: Anchor = Kyoto ──────────────────────────────────────────
shootout = compare_algorithms("Kyoto")

# ── Styled output ─────────────────────────────────────────────────────────────
styled = (
    shootout.style
    .format({
        "dice_score":           "{:.2f}",
        "tfidf_cosine_score":   "{:.2f}",
        "knn_similarity_score": "{:.2f}",
    })
    .background_gradient(subset=["dice_score"],           cmap="YlOrRd")
    .background_gradient(subset=["tfidf_cosine_score"],   cmap="YlGnBu")
    .background_gradient(subset=["knn_similarity_score"], cmap="PuBuGn")
    .set_caption("Algorithm Shootout — Anchor: Kyoto | Pure Thematic Similarity (0–100)")
)
styled

,region,shared_tags_count,dice_score,tfidf_cosine_score,knn_similarity_score
0,Kanazawa,9,64.29,61.09,65.63
1,Kamakura,7,50.00,35.12,55.67
2,Nara,7,50.00,32.45,56.41
3,Nikko,6,42.86,28.58,49.35
4,Hiroshima,5,35.71,21.80,41.92
5,Shirakawa-go,5,35.71,21.26,40.78
6,Okayama,5,37.04,20.63,40.72
7,Matsumoto,5,35.71,20.32,40.79
8,Shimane,4,29.63,15.98,29.81
9,Matsuyama,4,29.63,14.78,36.05


---
## Interpretation — Top-5 Differences Across Algorithms

### What the Scores Reveal

| Observation | Detail |
|---|---|
| **Kanazawa dominates all three** | Shares the highest raw tag count with Kyoto (`cherry_blossom`, `historic`, `traditional`, `geisha`, `matcha`, `crafts`, `culture`, `serene`, `architecture`, `autumn_leaves`). All algorithms agree — this is the strongest thematic match. |
| **TF-IDF re-ranks subtle matches** | TF-IDF elevates destinations with *rare* shared tags. A location sharing `geisha` or `matcha` with Kyoto scores higher than one sharing only `historic` — because `historic` appears in 15+ locations and has near-zero IDF weight, while `geisha` appears in only 2 locations. |
| **KNN captures non-tag dimensions** | KNN incorporates `cultural_score`, `experiential_score`, etc. — so a destination may rank higher in KNN despite fewer shared tags if its numerical profile closely mirrors Kyoto's. This explains why high-cultural-score destinations can leapfrog tag-rich but numerically dissimilar ones. |
| **Dice is generous but undiscriminating** | Dice consistently produces higher scores than TF-IDF for the same pair, because it weights all tags equally. A destination sharing 4 common tags (`historic`, `nature`, `temple`, `culture`) scores well even if those tags appear everywhere — Dice cannot distinguish between a distinctive match and a generic one. |
| **The gap between Dice and TF-IDF reveals tag-quality** | When Dice scores high but TF-IDF scores low, the match is *superficially* similar — driven by ubiquitous tags. When both score high, the match is *genuinely* distinctive. This gap metric itself could serve as a "match quality" signal in the final DSS. |

---

### Recommendation for the DSS Pipeline

**TF-IDF + Cosine should replace Jaccard/Dice as the primary thematic filter.** Its IDF weighting naturally penalises over-represented tags, producing rankings that better reflect *why* a tourist chose a specific anchor. KNN is the strongest candidate for a **secondary re-ranking layer** — once TF-IDF filters for thematic relevance, KNN can re-rank survivors by overall destination profile similarity (incorporating infrastructure, cleanliness, experiential quality, etc.).

Dice remains a useful **sanity-check baseline**: if TF-IDF ever ranks a destination below Dice by more than 30 points, the IDF weighting may be too aggressive and warrants manual tag review.